# The same Sobel filter three ways, on an AUP-ZU3 (Zynq UltraScale+ ZU3EG)

One algorithm - BT.601 luma followed by a 3x3 Sobel magnitude - implemented
three times and run back to back in this notebook:

| # | where it runs | written in |
|---|---------------|------------|
| 1 | **PS** - the four Cortex-A53 cores | Python / NumPy |
| 2 | **PL** - the FPGA fabric | C++ compiled by Vitis HLS |
| 3 | **PL** - the FPGA fabric | hand-written SystemVerilog (or VHDL) |

They run in that order on purpose. The software version comes first because it
is the *specification*: it is small enough to read in one sitting, it is the
thing you would write if you had no FPGA at all, and everything after it is
checked bit-for-bit against its output. Only once you have a golden answer and
a baseline number does an accelerator mean anything - "11 ms" on its own is
not a result.

The two hardware versions present an identical AXI4-Lite register map, so a
single driver class drives both; swapping implementations is a different
`.bit` file and nothing else.

Timings for every stage are collected in `RESULTS` as we go and summarised in
section 6.

**Before running:** this notebook needs `test.jpg` plus a bitstream/`.hwh`
pair per implementation. It looks for them in the usual build output
directories and in `/home/xilinx/ch11_hil/{hls,sv,vhdl}/` where `deploy.sh`
puts them.

It also needs the PL to itself. **If you have never deployed the DisplayPort
demo, there is nothing to do here** - skip to the end of this list. It is an
optional extra that drives an attached monitor, and this notebook neither needs
nor uses it.

If you *do* have it installed, stop it before running anything below. It
reprograms the fabric and drives this same accelerator on a timer, which
corrupts every measurement here:

```bash
sudo systemctl stop ch11-dp     # then re-run this notebook from the top
sudo systemctl start ch11-dp    # afterwards, to get the monitor back
```

Section 0 tells you which of those three situations you are in, so you do not
have to know in advance.

### A note on the Mali GPU

The ZU3EG *does* contain an ARM Mali-400 MP2 in the processing system, but it
plays no part in this flow, and it can't:

- **HLS targets the PL, not the PS.** Vitis HLS compiles C/C++ into RTL for the
  FPGA fabric. The Mali is a hardened block in the processor subsystem - there is
  no path from HLS to it.
- **Mali-400 has no compute API.** It supports OpenGL ES 1.1/2.0 and OpenVG 1.1
  only. No OpenCL, no compute shaders, so no general-purpose image processing.
- **Jupyter renders in your browser.** Anything the notebook displays is encoded
  on the board, sent over the network, and drawn by the client machine. The
  board's GPU and DisplayPort output are not in that path.

So "software" here means the A53s, and that is the honest comparison anyway:
the Sobel is a memory-bound integer stencil, which is exactly the shape of
problem the PL is good at and the shape a small mobile GPU is not.

## 0. Setup

Imports, the measurement harness, and where to find the bitstreams.

`bench()` runs a warm-up call, then `REPS` timed calls, and keeps the
**best** and the **median**. Best-of-N is the right statistic for a
deterministic kernel: the noise on a shared Linux box is one-sided, so the
fastest run is the one least polluted by the scheduler. The median is
reported alongside it as an honesty check - if the two are far apart, the
measurement is not stable and the number should not be trusted.

In [ ]:
import os
import time

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline

from pynq import Overlay, allocate

MODE_GRAY, MODE_SOBEL, MODE_INVERT = 0, 1, 2
MODE_NAMES = {MODE_GRAY: "grayscale", MODE_SOBEL: "sobel", MODE_INVERT: "invert"}

REPS = 5          # timed repetitions after one warm-up call

# Every measurement in this notebook is appended here and summarised in S6.
RESULTS = []


def record(impl, stage, seconds, pixels, exact=None, note=""):
    """Log one measurement and print it as it happens."""
    row = dict(impl=impl, stage=stage, seconds=seconds, pixels=pixels,
               ms=seconds * 1e3, mpix_s=pixels / seconds / 1e6,
               exact=exact, note=note)
    RESULTS.append(row)
    flag = "" if exact is None else ("  [bit-exact]" if exact else "  [APPROXIMATE]")
    print(f"{impl:<10} {stage:<26} {row['ms']:>10.2f} ms  "
          f"{row['mpix_s']:>8.2f} Mpixel/s{flag}")
    return row


def bench(fn, reps=REPS):
    """Warm up once, then time `reps` calls. Returns (best, median) seconds."""
    fn()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return min(ts), sorted(ts)[len(ts) // 2]


# Search order per implementation: local build output first, then wherever
# deploy.sh dropped it on the board, then the notebook's own directory.
IMPL_DIRS = {
    "hls":  ["out", "HLS/out", "../HLS/out", "/home/xilinx/ch11_hil/hls", "."],
    "sv":   ["out_sv", "../out_sv", "/home/xilinx/ch11_hil/sv", "sv"],
    "vhdl": ["out_vhdl", "../out_vhdl", "/home/xilinx/ch11_hil/vhdl", "vhdl"],
}


def bitstream(impl):
    """Locate image_filter.bit for `impl`; the .hwh must sit beside it."""
    for d in IMPL_DIRS[impl]:
        bit = os.path.join(d, "image_filter.bit")
        if os.path.exists(bit) and os.path.exists(bit[:-4] + ".hwh"):
            return bit
    raise FileNotFoundError(
        f"no image_filter.bit/.hwh for '{impl}' in {IMPL_DIRS[impl]}")


# Which of the two hand-written RTL implementations to run in section 5.
# "vhdl" is a drop-in swap - same register map, same block design, same result.
HDL_IMPL = "sv"

for impl in ("hls", HDL_IMPL):
    print(f"{impl:>5}: {bitstream(impl)}")


def check_pl_is_ours():
    """Report whether anything else is driving the accelerator.

    The ch11-dp DisplayPort demo runs this same IP on a timer and reprograms
    the PL, so it has to be stopped before any number below means anything.
    There are three outcomes and only one of them needs action - a new user who
    never deployed the demo should just see "not installed" and carry on.

    Note that `systemctl is-active` reports a unit that was never installed as
    "inactive", exactly like one you stopped yourself, so it cannot be the test
    on its own - hence the list-unit-files check first."""
    import subprocess

    def sysctl(*args):
        try:
            r = subprocess.run(("systemctl",) + args,
                               capture_output=True, text=True, timeout=5)
            return r.stdout.strip()
        except (FileNotFoundError, subprocess.SubprocessError, OSError):
            return None

    installed = sysctl("list-unit-files", "ch11-dp.service", "--no-legend")

    if installed is None:
        print("PL check: no systemd here. If anything else on this board drives\n"
              "          the accelerator or reprograms the PL, stop it first.")
        return

    if not installed:
        print("PL check: ch11-dp is not installed - the PL is yours. Nothing to do.\n"
              "          (That service is optional. It drives a DisplayPort\n"
              "           monitor and is not needed by this notebook.)")
        return

    state = sysctl("is-active", "ch11-dp")

    if state == "active":
        print("\n*** WARNING: ch11-dp is RUNNING. It drives this same IP on a\n"
              "    timer, so every number below will be wrong and its display\n"
              "    will break when this notebook reprograms the PL. Run\n"
              "        sudo systemctl stop ch11-dp\n"
              "    and re-run this notebook from the top. Afterwards:\n"
              "        sudo systemctl start ch11-dp\n"
              "    puts the monitor output back. ***\n")
        return

    msg = f"PL check: ch11-dp is installed but {state} - that is the state you want."
    if sysctl("is-enabled", "ch11-dp") == "enabled":
        msg += ("\n          It is enabled, so it will start again after a reboot;"
                "\n          stop it again before re-running these benchmarks.")
    print(msg)


check_pl_is_ours()

## 1. Load the JPEG

Decoding happens on the A53 cores via Pillow. Converting to RGBA gives 4 bytes
per pixel, which matches the accelerator's packed 32-bit word, so the DMA can
treat the frame as a flat array of `uint32` with no repacking on either side.

`MAX_WIDTH` is the depth of the line buffer the hardware was built with. A
wider image would run off the end of that BRAM, so it is a hard limit on the
frame the accelerator can accept, not a preference.

In [ ]:
MAX_WIDTH = 1920          # must match MAX_WIDTH in the HLS/RTL build

img = Image.open("test.jpg").convert("RGBA")

if img.width > MAX_WIDTH:
    scale = MAX_WIDTH / img.width
    img = img.resize((MAX_WIDTH, int(img.height * scale)), Image.LANCZOS)

W, H = img.size
NPIX = W * H
rgba_in = np.ascontiguousarray(np.array(img))     # (H, W, 4) uint8

print(f"{W} x {H} = {NPIX/1e6:.2f} Mpixel, {rgba_in.nbytes/2**20:.1f} MiB as RGBA")

plt.figure(figsize=(6, 6 * H / W))
plt.imshow(img); plt.axis("off"); plt.title("input"); plt.show()

---
# Part 1 - Software on the PS

## 2. The algorithm, spelled out

Everything below - Python, NumPy, HLS C++ and RTL - computes exactly this:

**Luma.** BT.601 in Q8 fixed point. The float coefficients 0.299 / 0.587 /
0.114 become the integers 77 / 150 / 29, and the divide by 256 is a right
shift:

$$Y = \frac{77R + 150G + 29B}{256}$$

Integer arithmetic is not an approximation forced on us by the hardware - it
is a *choice made in the software first*, so that the hardware can be
bit-exact rather than merely close. There is no rounding term: the shift
truncates, and every implementation truncates the same way.

**Sobel.** Two 3x3 convolutions on the luma plane,

$$G_x = \begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix} * Y
\qquad
G_y = \begin{bmatrix}-1&-2&-1\\0&0&0\\1&2&1\end{bmatrix} * Y$$

combined as $|G_x| + |G_y|$ and clamped to 255. The true magnitude is
$\sqrt{G_x^2+G_y^2}$; the absolute-value sum is the standard cheap stand-in,
and it is what the hardware does, so it is what the software must do.

**Borders.** The 3x3 window is undefined on the outermost ring of pixels, so
that one-pixel frame is forced to black. Silently replicating or wrapping
edge pixels would be defensible too - but the hardware has to pick one
behaviour, and a black frame is the one that is impossible to mistake for
real edge data.

**Output.** The 8-bit result is replicated across R, G and B with alpha
forced to 0xFF, so the output buffer has the same layout as the input.

### 2.1 Straight Python

The literal transcription: nested loops, one pixel at a time, no library doing
the work behind your back. This is the version to read if you want to know
what the hardware is doing - the HLS kernel in `src/image_filter.cpp` is
recognisably the same code with a line buffer bolted on.

It is also unusably slow, which is the point. Keep it in view while reading
the numbers at the end.

In [ ]:
def sobel_python(rgba, mode=MODE_SOBEL):
    """Pure-Python reference. Same arithmetic as the hardware, one pixel at a time."""
    H_, W_ = rgba.shape[0], rgba.shape[1]
    src = rgba.tolist()                       # list-of-lists is far faster to index

    # --- BT.601 luma, Q8 -------------------------------------------------
    gray = [[0] * W_ for _ in range(H_)]
    for r in range(H_):
        srow, grow = src[r], gray[r]
        for c in range(W_):
            px = srow[c]
            grow[c] = (77 * px[0] + 150 * px[1] + 29 * px[2]) >> 8

    # --- the filter itself ----------------------------------------------
    if mode == MODE_GRAY:
        out = gray
    elif mode == MODE_INVERT:
        out = [[255 - v for v in row] for row in gray]
    else:
        out = [[0] * W_ for _ in range(H_)]   # black border, filled in below
        for r in range(1, H_ - 1):
            up, mid, dn, orow = gray[r-1], gray[r], gray[r+1], out[r]
            for c in range(1, W_ - 1):
                gx = (up[c+1] + 2 * mid[c+1] + dn[c+1]) - \
                     (up[c-1] + 2 * mid[c-1] + dn[c-1])
                gy = (dn[c-1] + 2 * dn[c] + dn[c+1]) - \
                     (up[c-1] + 2 * up[c] + up[c+1])
                m = (gx if gx >= 0 else -gx) + (gy if gy >= 0 else -gy)
                orow[c] = 255 if m > 255 else m

    # --- replicate to RGBA ----------------------------------------------
    o = np.array(out, dtype=np.uint8)
    return np.dstack([o, o, o, np.full(o.shape, 255, np.uint8)])

### 2.2 NumPy

The same computation expressed as whole-array operations. Nothing is
approximated: the nine shifted slices of the luma plane are the nine taps of
the 3x3 window, `int32` throughout so no intermediate can overflow, and the
same truncating `>> 8`. It produces the identical array the Python loops do -
we check that below rather than asserting it.

This is the honest software baseline. Comparing an accelerator against triple-
nested Python would flatter it by two orders of magnitude; comparing it
against vectorised NumPy is the number worth quoting.

In [ ]:
def sobel_numpy(rgba, mode=MODE_SOBEL):
    """Vectorised reference. Bit-identical to sobel_python, ~1000x faster."""
    R = rgba[:, :, 0].astype(np.int32)
    G = rgba[:, :, 1].astype(np.int32)
    B = rgba[:, :, 2].astype(np.int32)
    gray = (77 * R + 150 * G + 29 * B) >> 8            # BT.601 luma, Q8

    if mode == MODE_GRAY:
        o = gray
    elif mode == MODE_INVERT:
        o = 255 - gray
    else:
        p = gray
        # the nine window taps as shifted views - no copies, no Python loop
        gx = (p[:-2, 2:] + 2 * p[1:-1, 2:] + p[2:, 2:]) - \
             (p[:-2, :-2] + 2 * p[1:-1, :-2] + p[2:, :-2])
        gy = (p[2:, :-2] + 2 * p[2:, 1:-1] + p[2:, 2:]) - \
             (p[:-2, :-2] + 2 * p[:-2, 1:-1] + p[:-2, 2:])
        o = np.zeros_like(gray)                         # black border
        o[1:-1, 1:-1] = np.clip(np.abs(gx) + np.abs(gy), 0, 255)

    o = o.astype(np.uint8)
    return np.dstack([o, o, o, np.full(o.shape, 255, np.uint8)])

### 2.3 Do the two agree?

On a small crop, because the Python version is measured in pixels per
millisecond rather than megapixels per second.

In [ ]:
CROP_W, CROP_H = 256, 192
r0, c0 = (H - CROP_H) // 2, (W - CROP_W) // 2
crop = np.ascontiguousarray(rgba_in[r0:r0+CROP_H, c0:c0+CROP_W])

for mode in (MODE_GRAY, MODE_SOBEL, MODE_INVERT):
    a = sobel_python(crop, mode)
    b = sobel_numpy(crop, mode)
    same = np.array_equal(a, b)
    print(f"{MODE_NAMES[mode]:<10} python == numpy : {same}")
    assert same, "the two software implementations disagree"

### 2.4 Time the software

The Python version is timed on the crop and the full-frame cost extrapolated,
because running it over 3 Mpixel would take minutes and tell us nothing we
cannot get by multiplying. The extrapolation is marked as such in the summary
and never used as a speed-up denominator.

NumPy is timed on the real frame.

In [ ]:
# --- pure Python, on the crop, extrapolated ----------------------------
t_best, t_med = bench(lambda: sobel_python(crop, MODE_SOBEL), reps=1)
crop_pix = CROP_W * CROP_H
print(f"pure Python on {CROP_W}x{CROP_H} crop: {t_best*1e3:.1f} ms "
      f"({crop_pix/t_best/1e6:.4f} Mpixel/s)")

t_py_full = t_best * NPIX / crop_pix        # extrapolated, not measured
record("PS python", "sobel (extrapolated)", t_py_full, NPIX, exact=True,
       note=f"measured on a {CROP_W}x{CROP_H} crop and scaled")
print(f"  -> a full {W}x{H} frame would take about {t_py_full:.1f} s")

In [ ]:
# --- NumPy, full frame -------------------------------------------------
for mode in (MODE_SOBEL, MODE_GRAY, MODE_INVERT):
    best, med = bench(lambda m=mode: sobel_numpy(rgba_in, m))
    row = record("PS numpy", f"{MODE_NAMES[mode]}", best, NPIX, exact=True)
    row["median_ms"] = med * 1e3

sw_sobel = sobel_numpy(rgba_in, MODE_SOBEL)     # the golden result, kept for §4/§5

### 2.5 OpenCV, for scale

OpenCV is what you would actually reach for on the PS, and it is
multi-threaded across all four A53s with NEON underneath. It is included so
the accelerator is measured against the *best* software on the board rather
than the most convenient.

Watch the one-thread and four-thread numbers: they land close together,
because a 3x3 integer stencil over 12 MiB is bound by memory bandwidth long
before it is bound by arithmetic, and the four A53s share one path to DDR.
That is the same wall the accelerator eventually hits, and it is worth seeing
it in software first.

It is labelled approximate, and that label is not a formality: `cvtColor` uses
different luma coefficients and rounds rather than truncates, `Sobel` uses
replicated borders, and `convertScaleAbs` saturates each gradient separately
before the add. Its output is close to ours but not equal to it, so it is
timed and never used as a correctness reference.

In [ ]:
try:
    import cv2
    rgb_in = np.ascontiguousarray(rgba_in[:, :, :3])

    def opencv_sobel(rgb, mode=MODE_SOBEL):
        g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        if mode == MODE_GRAY:
            return g
        if mode == MODE_INVERT:
            return cv2.bitwise_not(g)
        gx = cv2.Sobel(g, cv2.CV_16S, 1, 0, ksize=3)
        gy = cv2.Sobel(g, cv2.CV_16S, 0, 1, ksize=3)
        return cv2.convertScaleAbs(cv2.add(cv2.convertScaleAbs(gx),
                                           cv2.convertScaleAbs(gy)))

    print(f"OpenCV {cv2.__version__}, {cv2.getNumThreads()} threads")
    best, med = bench(lambda: opencv_sobel(rgb_in, MODE_SOBEL))
    record("PS opencv", "sobel (all cores)", best, NPIX, exact=False,
           note="different coefficients/rounding/borders")

    cv2.setNumThreads(1)
    best1, _ = bench(lambda: opencv_sobel(rgb_in, MODE_SOBEL))
    record("PS opencv", "sobel (1 core)", best1, NPIX, exact=False,
           note="single-threaded, for scaling reference")
    cv2.setNumThreads(0)
except ImportError:
    print("OpenCV not installed - skipping (nothing else depends on it)")

### 2.6 The software result

This is the image every hardware implementation now has to reproduce exactly.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5 * H / W))
ax[0].imshow(rgba_in);  ax[0].set_title("input JPEG");            ax[0].axis("off")
ax[1].imshow(sw_sobel); ax[1].set_title("PS Sobel (NumPy)");      ax[1].axis("off")
plt.tight_layout(); plt.show()

---
# Part 2 - The HLS accelerator in the PL

## 3. Load the overlay and set up the driver

`src/image_filter.cpp` is the C++ that Vitis HLS turned into this IP. It is the
software above restructured for hardware: a three-stage `DATAFLOW` pipeline
(read+greyscale, 3x3 window over a two-row line buffer, replicate+write) with
`PIPELINE II=1` on every loop, so it retires one pixel per clock once the
pipeline is full.

The line buffer is the whole trick. Software indexes `gray[r-1][c+1]` freely
because DDR is random-access; hardware cannot afford to re-read each pixel
three times, so two rows are held in BRAM and the 3x3 window slides across
them. Every pixel is read from DDR exactly once.

In [ ]:
class FilterAccel:
    """Driver for the image_filter IP. Identical for the HLS and the RTL builds -
    both present the same AXI4-Lite register map at the same offsets."""

    def __init__(self, bitfile):
        self.bitfile = bitfile
        self.ol = Overlay(bitfile)
        self.ip = self.ol.image_filter_0

    def configure(self, src_buf, dst_buf, width, height, mode):
        rm = self.ip.register_map
        sp, dp = src_buf.physical_address, dst_buf.physical_address
        rm.src_1 = sp & 0xFFFFFFFF
        rm.src_2 = (sp >> 32) & 0xFFFFFFFF
        rm.dst_1 = dp & 0xFFFFFFFF
        rm.dst_2 = (dp >> 32) & 0xFFFFFFFF
        rm.img_width  = width
        rm.img_height = height
        rm.mode       = mode

    def compute(self, src_buf, dst_buf, width, height, mode, timeout=200_000_000):
        """Configure, start, poll ap_done. No cache maintenance - PL time only."""
        self.configure(src_buf, dst_buf, width, height, mode)
        rm = self.ip.register_map
        rm.CTRL.AP_START = 1
        n = 0
        while rm.CTRL.AP_DONE == 0:
            n += 1
            if n > timeout:
                raise RuntimeError(f"ap_done never asserted (mode={mode})")

    def run(self, src_buf, dst_buf, width, height, mode):
        """End to end: flush the source out of the A53 caches, run, invalidate
        the destination so the CPU does not read stale lines back."""
        src_buf.flush()
        self.compute(src_buf, dst_buf, width, height, mode)
        dst_buf.invalidate()


hls_accel = FilterAccel(bitstream("hls"))
print(f"loaded {hls_accel.bitfile}\n")
print(hls_accel.ip.register_map)

The register map above tells you the exact names HLS generated. The 64-bit
pointer arguments are split into two 32-bit registers each - `src_1`/`src_2`
and `dst_1`/`dst_2`.

Note the scalar arguments are `img_width`/`img_height`, not `width`/`height`.
That is deliberate. PYNQ builds this register map by creating a property per
register field on its `Register` class, and `Register.__init__` assigns
`self.width` - so a field named `width` shadows it with a property and the
next attribute read recurses forever:

```
RecursionError: maximum recursion depth exceeded
```

...raised by this very cell, before the accelerator is ever started. The
reserved field names are `address`, `width`, `debug` and `access`.

## 3.1 Contiguous buffers

`allocate()` returns physically-contiguous DMA-capable memory and exposes its
physical address, which is what the accelerator's AXI master needs. A NumPy
array from `np.zeros` will not do: it is virtual, scattered across pages the
IP has no MMU to translate, and the PL would happily DMA over whatever else
lives in those physical frames.

The same two buffers are reused by both hardware implementations - CMA is a
scarce resource and 12 MiB a side is not nothing.

In [ ]:
src_buf = allocate(shape=(H, W, 4), dtype=np.uint8)
dst_buf = allocate(shape=(H, W, 4), dtype=np.uint8)
src_buf[:] = rgba_in

print(f"src: 0x{src_buf.physical_address:012X}  "
      f"dst: 0x{dst_buf.physical_address:012X}  "
      f"({src_buf.nbytes/2**20:.1f} MiB each)")

## 3.2 Run it, and check it against the software

Correctness before speed. `hw == sw_sobel` has to be exactly true - not "close",
not "PSNR of 48 dB". Both sides do integer arithmetic with the same
truncation and the same border rule, so any difference at all is a bug in the
hardware, and a single wrong pixel is worth more as a debugging signal than
any timing number.

In [ ]:
hls_accel.run(src_buf, dst_buf, W, H, MODE_SOBEL)
hw_sobel = np.array(dst_buf)

diff = np.abs(hw_sobel.astype(np.int16) - sw_sobel.astype(np.int16))
n_wrong = int((diff != 0).any(axis=2).sum())
print(f"max abs difference : {diff.max()}")
print(f"pixels differing   : {n_wrong} of {NPIX}")
print("HLS vs software    :", "BIT-EXACT" if n_wrong == 0 else "*** MISMATCH ***")

## 3.3 Time it

Three numbers, because "how fast is it" has three different honest answers.

**Compute only** - write seven registers over AXI4-Lite, set `ap_start`, poll
`ap_done`. This is the accelerator's own throughput and nothing else.

**End to end** adds `flush()` and `invalidate()`. The A53 caches do not snoop
these AXI ports, so the source has to be pushed out of L1/L2 before the PL
reads it, and the stale destination lines dropped before the CPU reads them.
Both are mandatory for correctness - skip either one and you get garbage, not
a slightly wrong answer.

**Frame in and out** is what an application actually pays: copy a new frame
into the CMA buffer, run, and copy the result back out into an ordinary NumPy
array. No accelerator makes those two 11 MiB memcpys go away, and they are the
reason "the kernel takes 16 ms" and "the pipeline takes 16 ms" are different
claims.

Watch what the gap between the first two turns out to be - it is smaller than
you would guess, and the reason is discussed in section 6.

In [ ]:
for mode in (MODE_SOBEL, MODE_GRAY, MODE_INVERT):
    best, med = bench(lambda m=mode: hls_accel.compute(src_buf, dst_buf, W, H, m))
    row = record("PL hls", f"{MODE_NAMES[mode]} (compute)", best, NPIX,
                 exact=True if mode == MODE_SOBEL else None)
    row["median_ms"] = med * 1e3

best, med = bench(lambda: hls_accel.run(src_buf, dst_buf, W, H, MODE_SOBEL))
row = record("PL hls", "sobel (end to end)", best, NPIX, exact=True,
             note="+ cache flush and invalidate")
row["median_ms"] = med * 1e3


def frame_in_out(accel, mode=MODE_SOBEL):
    """What an application really pays per frame: copy in, run, copy out."""
    src_buf[:] = rgba_in                 # a fresh frame arrives from the CPU
    accel.run(src_buf, dst_buf, W, H, mode)
    return np.array(dst_buf)             # and the result leaves CMA


best, med = bench(lambda: frame_in_out(hls_accel))
row = record("PL hls", "sobel (frame in + out)", best, NPIX, exact=True,
             note="+ two 11 MiB memcpys")
row["median_ms"] = med * 1e3

## 3.4 All three modes

Grayscale and invert share the entire datapath with Sobel - same read, same
line buffer, same write - and differ only in which value the window stage
emits. In software those would be three functions; in hardware they are one
pipeline and a 2-bit mode register, because the fabric is spent whether you
use it or not.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5 * H / W))
for a, mode in zip(ax, (MODE_GRAY, MODE_SOBEL, MODE_INVERT)):
    hls_accel.run(src_buf, dst_buf, W, H, mode)
    a.imshow(np.array(dst_buf))
    a.set_title(f"HLS {MODE_NAMES[mode]}")
    a.axis("off")
plt.tight_layout(); plt.show()

---
# Part 3 - The hand-written RTL accelerator in the PL

## 4. Reload the PL with the SystemVerilog build

Same block design, same clock, same register offsets - a different
`image_filter.bit`. `HDL_IMPL` at the top selects `"sv"` or `"vhdl"`; the two
RTL versions are line-for-line translations of each other and produce
identical bitstream behaviour, so running both would only prove that VHDL and
SystemVerilog can express the same logic.

The register map is not *similar* to the HLS one, it is byte-for-byte
identical - `image_filter_ctrl.sv` was written against the layout Vitis HLS
generates, which is why `FilterAccel` above needs no changes and why
`deploy.sh` can swap implementations under a running service.

Reloading the overlay reprograms the PL. The CMA buffers live in DDR and are
untouched, so they carry across.

One cosmetic difference will show up in the printout below: where the HLS
build's registers list named bit-fields, the RTL build's mostly show a single
`value`. That is the packaged IP's metadata, not the hardware - Vitis HLS
emits a field description per argument into the `.hwh` and the Vivado IP
packager has nothing to emit for a hand-written `.sv`. The offsets, widths and
semantics are identical, which is why the same `FilterAccel` drives it. `CTRL`
still comes up fully decoded because PYNQ knows that register by name.

In [ ]:
hdl_accel = FilterAccel(bitstream(HDL_IMPL))
print(f"loaded {hdl_accel.bitfile}  (implementation: {HDL_IMPL})\n")
print(hdl_accel.ip.register_map)

## 4.1 Three-way check

The RTL has to match the software *and* the HLS build. Two independent
implementations agreeing with a third that was written first, in a different
language, on a different machine, is about as much confidence as you can buy
without a formal proof.

In [ ]:
src_buf[:] = rgba_in
hdl_accel.run(src_buf, dst_buf, W, H, MODE_SOBEL)
rtl_sobel = np.array(dst_buf)

vs_sw  = int((rtl_sobel != sw_sobel).any(axis=2).sum())
vs_hls = int((rtl_sobel != hw_sobel).any(axis=2).sum())

print(f"{HDL_IMPL.upper()} RTL vs software : {vs_sw} pixels differ  "
      f"-> {'BIT-EXACT' if vs_sw == 0 else '*** MISMATCH ***'}")
print(f"{HDL_IMPL.upper()} RTL vs HLS      : {vs_hls} pixels differ  "
      f"-> {'BIT-EXACT' if vs_hls == 0 else '*** MISMATCH ***'}")

## 4.2 Time it

Expect the RTL to land within a percent or two of the HLS build. That is the
honest result, and it is the useful one: for a kernel this regular - a
one-pixel-per-clock streaming stencil - HLS finds the same schedule a human
would, and the runtime is set by DDR bandwidth and the AXI4-Lite handshakes
either way. Where hand-written RTL earns its cost is control-heavy logic, odd
interfaces and tight timing closure, not throughput on a well-behaved
pipeline.

In [ ]:
for mode in (MODE_SOBEL, MODE_GRAY, MODE_INVERT):
    best, med = bench(lambda m=mode: hdl_accel.compute(src_buf, dst_buf, W, H, m))
    row = record(f"PL {HDL_IMPL}", f"{MODE_NAMES[mode]} (compute)", best, NPIX,
                 exact=True if mode == MODE_SOBEL else None)
    row["median_ms"] = med * 1e3

best, med = bench(lambda: hdl_accel.run(src_buf, dst_buf, W, H, MODE_SOBEL))
row = record(f"PL {HDL_IMPL}", "sobel (end to end)", best, NPIX, exact=True,
             note="+ cache flush and invalidate")
row["median_ms"] = med * 1e3

best, med = bench(lambda: frame_in_out(hdl_accel))
row = record(f"PL {HDL_IMPL}", "sobel (frame in + out)", best, NPIX, exact=True,
             note="+ two 11 MiB memcpys")
row["median_ms"] = med * 1e3

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5 * H / W))
for a, (im, title) in zip(ax, [(sw_sobel,  "PS NumPy"),
                               (hw_sobel,  "PL HLS"),
                               (rtl_sobel, f"PL {HDL_IMPL.upper()} RTL")]):
    a.imshow(im); a.set_title(title); a.axis("off")
plt.suptitle("three implementations, one image - pixel for pixel identical")
plt.tight_layout(); plt.show()

---
## 5. Performance summary

Every measurement collected above, normalised to the NumPy Sobel - the
fastest *bit-exact* software on the board, and therefore the only fair
denominator. OpenCV is faster still but computes something slightly
different; the pure-Python figure is extrapolated. Both are marked.

In [ ]:
base = next(r for r in RESULTS
            if r["impl"] == "PS numpy" and r["stage"] == "sobel")
t_base = base["seconds"]

def vs_numpy(r):
    if r is base:
        return "baseline"
    s = t_base / r["seconds"]
    return f"{s:.1f}x faster" if s >= 1 else f"{1/s:.1f}x slower"


hdr = (f"{'where':<10} {'stage':<24} {'best ms':>10} {'median':>8} "
       f"{'Mpixel/s':>9} {'vs NumPy':>13}  notes")
print(hdr)
print("-" * len(hdr))
for r in RESULTS:
    tag = "" if r["exact"] is not False else "approx; "
    med = f"{r['median_ms']:.2f}" if "median_ms" in r else "-"
    print(f"{r['impl']:<10} {r['stage']:<24} {r['ms']:>10.2f} {med:>8} "
          f"{r['mpix_s']:>9.2f} {vs_numpy(r):>13}  {tag}{r['note']}")
print("-" * len(hdr))
print(f"frame: {W}x{H} = {NPIX/1e6:.2f} Mpixel   |   baseline: PS numpy sobel "
      f"= {t_base*1e3:.2f} ms")

In [ ]:
# --- headline comparison: Sobel, like for like ------------------------
def pick(impl, stage):
    for r in RESULTS:
        if r["impl"] == impl and r["stage"] == stage:
            return r
    return None

bars = [
    ("pure Python\n(extrapolated)", pick("PS python", "sobel (extrapolated)")),
    ("NumPy\n(PS, 1 core)",         pick("PS numpy",  "sobel")),
    ("OpenCV\n(PS, 4 cores)",       pick("PS opencv", "sobel (all cores)")),
    ("HLS\n(PL, compute)",          pick("PL hls",    "sobel (compute)")),
    (f"{HDL_IMPL.upper()} RTL\n(PL, compute)", pick(f"PL {HDL_IMPL}", "sobel (compute)")),
    ("HLS\n(PL, end to end)",       pick("PL hls",    "sobel (end to end)")),
    ("HLS\n(PL, frame in+out)",     pick("PL hls",    "sobel (frame in + out)")),
]
bars = [(l, r) for l, r in bars if r is not None]

labels = [l for l, _ in bars]
tput   = [r["mpix_s"] for _, r in bars]
colors = ["#b0b0b0" if r["impl"].startswith("PS") else "#1f77b4" for _, r in bars]

fig, ax = plt.subplots(figsize=(11, 4.5))
b = ax.bar(labels, tput, color=colors)
ax.set_yscale("log")
ax.set_ylabel("Mpixel/s  (log scale)")
ax.set_title(f"Sobel throughput, {W}x{H} - grey = PS (software), blue = PL (fabric)")
ax.grid(axis="y", which="both", alpha=0.3)
for rect, (_, r) in zip(b, bars):
    ax.annotate(f"{r['mpix_s']:.2f}\n{r['ms']:.1f} ms",
                (rect.get_x() + rect.get_width() / 2, rect.get_height()),
                textcoords="offset points", xytext=(0, 4),
                ha="center", fontsize=9)
plt.tight_layout(); plt.show()

### Reading the numbers

**The log scale is doing a lot of work.** Pure Python to fabric spans more
than three orders of magnitude, and no linear axis can show that *and* the
HLS-vs-RTL difference on the same plot. Note where the win comes from: the
jump from Python loops to NumPy is far larger than the jump from NumPy to the
FPGA. Most of the speed available to you is won before any hardware is
involved, and an accelerator benchmarked against unvectorised software is an
accelerator whose speed-up figure means nothing.

**HLS and hand-written RTL land on top of each other**, within about one
percent - and which one wins is inside the run-to-run noise. For a streaming
stencil at one pixel per clock, both are limited by the same DDR bandwidth and
the same AXI4-Lite polling overhead, so there is simply no room for a clever
schedule to show up. If your kernel looks like this one, HLS costs you nothing
in performance. Where hand-written RTL earns its keep is control-heavy logic,
awkward external interfaces, and timing closure - not throughput on a
well-behaved pipeline.

**Cache maintenance is cheap; copying is not.** "Compute" to "end to end"
barely moves, which is easy to misread as "cache coherence does not matter" -
it does, and dropping either call gives you garbage. It is cheap *here* for a
specific reason: the buffers are 11 MiB against a 1 MiB L2, the source is
written once and never re-dirtied inside the timing loop, so by the time
`flush()` runs there is almost nothing dirty left to write back.

"Frame in + out" is the row that hurts, and it contains no filtering at all -
just two 11 MiB memcpys moving the frame into CMA and the result back out.
That is the cost of treating the accelerator as a function you call with
ordinary arrays. The fix is never a faster kernel; it is not moving the data.
Keep the frame in CMA for the whole pipeline, hand the PL the buffer the
capture stage already filled, or let it write straight into the DisplayPort
framebuffer - all of which drop this row to zero.

**Polling `ap_done` burns a core.** The `while rm.CTRL.AP_DONE == 0` loop is
an AXI4-Lite read per iteration, each one crossing into the PL. It keeps the
measurement clean and the code short, but an interrupt (the IP raises one,
and `GIER`/`IP_IER` are right there in the register map) frees the A53 to do
something useful while the fabric works.

**A single frame flatters software and hardware differently.** The PL number
includes per-call register setup that amortises to nothing on a video stream;
the software numbers benefit from a warm cache that a real 30 fps pipeline
would not have.

## 6. Free the buffers

Contiguous memory is a scarce resource - always release it. Nothing else on
the board can get a large CMA allocation until you do, and a notebook kernel
left running holds them indefinitely.

In [ ]:
src_buf.freebuffer()
dst_buf.freebuffer()
print("buffers released")